# 第一阶段 步骤06：手动进行反向传播

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第六步。

---

## 核心目标

把步骤5 的链式法则**变成代码**：给 `Variable` 加 `grad`（梯度），给 `Function` 加 `backward` 方法，然后**手动**按反向顺序调用，求出 $\frac{dy}{dx}$。

## 6.1 Variable 类的功能扩展

`Variable` 原来只存普通值 `data`，现在新增一个 `grad` 实例变量，用来存**导数值（梯度）**。

- `grad` 同样是 `numpy.ndarray` 类型；
- 初始化为 `None`，反向传播真正算导数时才被赋值。

## 6.2 Function 类的功能扩展

`Function` 新增两处：

1. **`backward(gy)` 方法** —— 反向传播的接口（基类里先抛 `NotImplementedError`）；
2. **保存输入** —— 在 `__call__` 里把输入存为 `self.input`，这样 `backward` 里能拿到输入。

In [ ]:
import numpy as np

# 6.1 Variable：新增 grad（梯度）
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None    # 存储梯度

# 6.2 Function：新增 backward 接口 + 保存输入 self.input
class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        self.input = input   # 保存输入变量，供 backward 使用
        return output

    def forward(self, x):
        raise NotImplementedError()

    def backward(self, gy):
        raise NotImplementedError()

# 6.3 具体函数实现 backward（局部导数 × 传来的 gy）
class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy      # dy/dx = 2x
        return gx

class Exp(Function):
    def forward(self, x):
        return np.exp(x)

    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy  # dy/dx = e^x
        return gx

## 6.3 Square 类和 Exp 类的 backward

每个具体函数只需在 `backward` 里写"**自身导数 × 传来的梯度**"：

| 函数 | 正向 | 自身导数（局部导数） |
| --- | --- | --- |
| Square | $y = x^2$ | $2x$ |
| Exp | $y = e^x$ | $e^x$ |

参数 `gy` 是**从输出端传来的梯度**，返回值 = `gy × 自身导数`，再继续往输入端传——这就是链式法则"连乘"在代码里的体现。

## 6.4 反向传播的实现

以 $y = (e^{x^2})^2$ 为例，按与正向**相反**的顺序（C → B → A）手动调用 `backward`：

1. 从输出 y 开始，令 `y.grad = 1`（y 对自身的导数是 1）；
2. 依次 `C.backward → B.backward → A.backward`，每个返回值存进对应输入变量的 `grad`；
3. 最终 `x.grad` 就是 $\frac{dy}{dx}$。

In [ ]:
# 正向传播
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

# 手动反向传播（逆序：C → B → A）
y.grad = np.array(1.0)          # dy/dy = 1
b.grad = C.backward(y.grad)
a.grad = B.backward(b.grad)
x.grad = A.backward(a.grad)

print(x.grad)   # 3.297442541400256

## 这一步的"为什么"

- **`backward` 为什么要乘 `gy`？** 链式法则要求"下游导数的连乘结果"一路往下传，`gy` 就是这个连乘结果。
- **`grad` 为什么是 `ndarray`？** 为将来支持向量 / 矩阵等多维输入做准备，每个维度都要有自己的导数。

## 局限：调用顺序还得靠手写

目前反向传播的调用顺序（C→B→A）是**手动写死**的，函数一多就极易出错、无法扩展。

---

> 预告：步骤7 会给 `Variable` 加上 `creator`（记录"谁创造了我"），让程序**自动**按正确顺序做反向传播。